# SmartReads: Title + Authors Recommender (Top 500)
Uses TF‑IDF on `title` + `authors` for similarity, based on the first 500 rows of the cleaned dataset.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv('data/raw/books_clean_500.csv', dtype=str)
df.head()

## Build TF‑IDF features over title + authors

In [ ]:
corpus = (df['title'].fillna('') + ' ' + df['authors'].fillna('')).values
tfidf = TfidfVectorizer(stop_words='english', max_features=8000, ngram_range=(1,2))
X = tfidf.fit_transform(corpus)
X.shape

## Recommend by title substring or free text

In [ ]:
def recommend(query, topk=5):
    match = df[df['title'].str.lower().str.contains(query.lower())]
    if match.empty:
        qv = tfidf.transform([query])
        sims = cosine_similarity(qv, X).ravel()
    else:
        import numpy as np
        vec = X[match.index.tolist()].mean(axis=0)
        sims = cosine_similarity(vec, X).ravel()
        sims[match.index.tolist()] = -1
    idxs = sims.argsort()[::-1][:topk]
    cols = [c for c in ['title','authors','average_rating','ratings_count','publication_date','publisher'] if c in df.columns]
    return df.iloc[idxs][cols].assign(score=sims[idxs])

recommend('harry', 5)

## Export helper

In [ ]:
def export_recs(query, path='reports/recs.csv', topk=10):
    import os
    os.makedirs('reports', exist_ok=True)
    recs = recommend(query, topk)
    recs.to_csv(path, index=False)
    return path

export_recs('love', 'reports/recs_love.csv', 8)